# 06 — Feed Comment Exploration

Explore the `feed_comment` object type from the raw Backyard dump.

**Goal:** understand which fields exist, their coverage, boolean/categorical distributions, nested structures (files, audiences, mentions), and relationships to existing graph entities — so we can finalise the ontology before building extractors, loaders, and the pipeline.

### Quick Stats (from initial scan)
| Metric | Value |
|---|---|
| Total records | 1 271 |
| Distinct keys | 25 |
| Unique authors | 228 (100 % overlap with People) |
| Unique feeds referenced | 686 (547 match raw feeds, 532 match normalised Post\|Recognition) |
| Unique sites referenced | 26 (100 % overlap with Site) |
| Unique contents referenced | 248 (100 % overlap with Content) |
| Records with file attachments | 65 |
| Records with audiences | 233 |
| Inline mentions (user+site) | 858 |

### Candidate Graph Model (to be validated)
| Relationship | Direction |
|---|---|
| `COMMENTED_ON` | `(FeedComment)→(Post\|Recognition)` |
| `AUTHORED` | `(People)→(FeedComment)` |
| `BELONGS_TO_SITE` | `(FeedComment)→(Site)` |
| `BELONGS_TO_CONTENT` | `(FeedComment)→(Content)` |
| `MENTIONS` | `(FeedComment)→(People\|Site)` |
| `HAS_FILE` | `(FeedComment)→(File)` |

## 0 · Setup

In [ ]:
import json
import sys
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import RAW_FILE, NORMALIZED_DIR

print(f"Project root      : {PROJECT_ROOT}")
print(f"Raw file          : {RAW_FILE}")
print(f"Raw file exists   : {RAW_FILE.exists()}")
print(f"Normalized dir    : {NORMALIZED_DIR}")
print(f"Normalized exists : {NORMALIZED_DIR.exists()}")

## 1 · Load Raw Data

In [ ]:
# Load all feed_comment records + collect entity IDs for cross-reference checks
feed_comments = []
people_ids = set()
site_ids = set()
content_ids = set()
file_ids = set()
raw_feed_ids = set()

with open(RAW_FILE) as f:
    for line in f:
        obj = json.loads(line)
        ot = obj.get("object_type")

        if ot == "feed_comment":
            feed_comments.append(obj)
        elif ot == "people" and obj.get("id"):
            people_ids.add(obj["id"])
        elif ot == "site" and obj.get("id"):
            site_ids.add(obj["id"])
        elif ot == "content" and obj.get("id"):
            content_ids.add(obj["id"])
        elif ot == "file" and obj.get("id"):
            file_ids.add(obj["id"])
        elif ot == "feed" and obj.get("id"):
            raw_feed_ids.add(obj["id"])

n = len(feed_comments)
print(f"Feed comment records      : {n}")
print(f"People IDs (raw)          : {len(people_ids)}")
print(f"Site IDs (raw)            : {len(site_ids)}")
print(f"Content IDs (raw)         : {len(content_ids)}")
print(f"File IDs (raw)            : {len(file_ids)}")
print(f"Feed IDs (raw)            : {len(raw_feed_ids)}")

In [ ]:
# Save 10 sample records for debugging
sample_path = PROJECT_ROOT / "data" / "debug" / "sample_feed_comments.jsonl"
sample_path.parent.mkdir(parents=True, exist_ok=True)
with open(sample_path, "w") as out:
    for item in feed_comments[:10]:
        out.write(json.dumps(item) + "\n")
print(f"Saved {min(10, n)} samples → {sample_path}")

# Pretty-print one sample
sample = dict(feed_comments[0])
for heavy_key in ["feed_comment_data", "feed_comment_html_data"]:
    if heavy_key in sample and isinstance(sample[heavy_key], str) and len(sample[heavy_key]) > 200:
        sample[heavy_key] = sample[heavy_key][:200] + "...[truncated]..."
print("\n--- Sample feed_comment record ---")
print(json.dumps(sample, indent=2, default=str))

## 2 · Key Presence & Null Analysis

In [ ]:
all_keys = set()
for item in feed_comments:
    all_keys.update(item.keys())

print(f"Total distinct keys in feed_comment records: {len(all_keys)}\n")
print(f"{'Key':<50} {'Present':>8} {'Null':>8} {'Non-empty':>10}")
print("-" * 80)
for k in sorted(all_keys):
    present = sum(1 for s in feed_comments if k in s)
    nulls = sum(1 for s in feed_comments if k in s and s.get(k) is None)
    non_empty = sum(1 for s in feed_comments if s.get(k) not in (None, '', [], {}))
    pct = f"{non_empty / n:.1%}" if n else "–"
    print(f"{k:<50} {present:>8} {nulls:>8} {non_empty:>8}  {pct}")

## 3 · Boolean & Categorical Field Distributions

In [ ]:
boolean_categorical_fields = [
    "is_deleted",
    "feed_comment_content_is_deleted",
    "feed_comment_site_is_active",
    "feed_comment_content_type",
    "feed_comment_site_type",
]

for field in boolean_categorical_fields:
    counts = Counter(str(s.get(field)) for s in feed_comments)
    print(f"\n{field}:")
    for k, v in counts.most_common(20):
        print(f"  {k}: {v}")

## 4 · Timestamp Analysis

In [ ]:
from datetime import datetime

def parse_ts(s):
    if not s:
        return None
    try:
        return datetime.fromisoformat(s.replace("Z", "+00:00"))
    except Exception:
        return None

created_dates = [parse_ts(c.get("createddate")) for c in feed_comments]
modified_dates = [parse_ts(c.get("lastmodifieddate")) for c in feed_comments]

created_valid = [d for d in created_dates if d]
modified_valid = [d for d in modified_dates if d]

print(f"createddate   — valid: {len(created_valid)}/{n}")
if created_valid:
    print(f"  min: {min(created_valid).isoformat()}")
    print(f"  max: {max(created_valid).isoformat()}")

print(f"\nlastmodifieddate — valid: {len(modified_valid)}/{n}")
if modified_valid:
    print(f"  min: {min(modified_valid).isoformat()}")
    print(f"  max: {max(modified_valid).isoformat()}")

# created == modified => never edited
same_ts = sum(1 for c in feed_comments if c.get("createddate") == c.get("lastmodifieddate"))
print(f"\ncreateddate == lastmodifieddate (never edited): {same_ts}/{n} ({same_ts/n:.1%})")

## 5 · Text Content Fields

In [ ]:
# feed_comment_data / feed_comment_html_data / feed_comment_ref_data comparison
data_present = sum(1 for c in feed_comments if c.get("feed_comment_data") not in (None, ""))
html_present = sum(1 for c in feed_comments if c.get("feed_comment_html_data") not in (None, ""))
ref_present = sum(1 for c in feed_comments if c.get("feed_comment_ref_data") not in (None, ""))

print(f"feed_comment_data      non-empty: {data_present}/{n}")
print(f"feed_comment_html_data non-empty: {html_present}/{n}")
print(f"feed_comment_ref_data  non-empty: {ref_present}/{n}")

# Length distributions
data_lens = [len(c.get("feed_comment_data", "") or "") for c in feed_comments]
html_lens = [len(c.get("feed_comment_html_data", "") or "") for c in feed_comments]
ref_lens = [len(c.get("feed_comment_ref_data", "") or "") for c in feed_comments]

df_lens = pd.DataFrame({
    "feed_comment_data_len": data_lens,
    "feed_comment_html_data_len": html_lens,
    "feed_comment_ref_data_len": ref_lens,
})
print("\nLength statistics:")
print(df_lens.describe().to_string())

In [ ]:
# Is feed_comment_ref_data always a plain-text version?
# Check a few records where both ref_data and html_data exist
for i, c in enumerate(feed_comments[:10]):
    ref = (c.get("feed_comment_ref_data") or "")[:100]
    html = (c.get("feed_comment_html_data") or "")[:100]
    if ref and html:
        print(f"\n--- Record {i} ---")
        print(f"  ref_data : {ref}")
        print(f"  html_data: {html}")

## 6 · Nested Structures

### 6.1 `feed_comment_file` (attachments)

In [ ]:
files_nonempty = [c for c in feed_comments if c.get("feed_comment_file") and len(c["feed_comment_file"]) > 0]
print(f"Records with non-empty feed_comment_file: {len(files_nonempty)}/{n}")

# File count distribution
file_counts = Counter(len(c.get("feed_comment_file") or []) for c in feed_comments)
print("\nFile count distribution:")
for cnt, freq in sorted(file_counts.items()):
    label = f"  {cnt} files" if cnt != 1 else "  1 file"
    print(f"{label}: {freq} records")

if files_nonempty:
    sample_file = files_nonempty[0]["feed_comment_file"][0]
    print(f"\nSample file item keys: {sorted(sample_file.keys())}")
    print(json.dumps(sample_file, indent=2, default=str))

In [ ]:
# File-level field analysis
if files_nonempty:
    all_file_items = []
    for c in feed_comments:
        for fi in (c.get("feed_comment_file") or []):
            if isinstance(fi, dict):
                all_file_items.append(fi)

    print(f"Total file attachment items: {len(all_file_items)}")

    # Boolean/categorical in file items
    for fk in ["file_is_image", "file_is_video", "file_type", "file_extension", "file_mime_type", "file_provider"]:
        vals = Counter(str(fi.get(fk)) for fi in all_file_items)
        print(f"\n{fk}:")
        for k, v in vals.most_common(15):
            print(f"  {k}: {v}")

    # Overlap with normalised File IDs
    comment_file_ids = set(fi.get("file_id") for fi in all_file_items if fi.get("file_id"))
    overlap = comment_file_ids & file_ids
    print(f"\nfile_id overlap with raw File objects: {len(overlap)}/{len(comment_file_ids)}")

### 6.2 `feed_audiences`

In [ ]:
aud_nonempty = [c for c in feed_comments if c.get("feed_audiences") and len(c["feed_audiences"]) > 0]
print(f"Records with non-empty feed_audiences: {len(aud_nonempty)}/{n}")

# Audience count distribution
aud_counts = Counter(len(c.get("feed_audiences") or []) for c in feed_comments)
print("\nAudience count distribution:")
for cnt, freq in sorted(aud_counts.items()):
    if cnt == 0:
        continue
    print(f"  {cnt} audiences: {freq} records")

# What are the audience values? (strings = IDs)
if aud_nonempty:
    all_aud_vals = []
    for c in feed_comments:
        for a in (c.get("feed_audiences") or []):
            all_aud_vals.append(a)

    sample_types = Counter(type(a).__name__ for a in all_aud_vals)
    print(f"\nAudience item types: {dict(sample_types)}")
    print(f"Total audience values: {len(all_aud_vals)}")
    print(f"Unique audience IDs: {len(set(all_aud_vals))}")
    print(f"Sample: {all_aud_vals[:5]}")

    # Check if audience IDs match any known entity
    aud_id_set = set(a for a in all_aud_vals if isinstance(a, str))
    for label, ids in [("People", people_ids), ("Site", site_ids), ("Content", content_ids)]:
        overlap = aud_id_set & ids
        if overlap:
            print(f"  Audience IDs overlapping with {label}: {len(overlap)}")

### 6.3 `feed_comment_deleted_files`

In [ ]:
del_nonempty = [c for c in feed_comments if c.get("feed_comment_deleted_files") and len(c["feed_comment_deleted_files"]) > 0]
print(f"Records with non-empty feed_comment_deleted_files: {len(del_nonempty)}/{n}")
if del_nonempty:
    print("Sample:", del_nonempty[0]["feed_comment_deleted_files"][:3])

## 7 · Inline Mentions (from `feed_comment_data` JSON doc)

In [ ]:
# Parse the JSON doc structure in feed_comment_data and extract mentions
mention_records = []  # (comment_id, mention_id, mention_label, mention_type)

for c in feed_comments:
    data_str = c.get("feed_comment_data", "")
    try:
        doc = json.loads(data_str)
    except Exception:
        continue

    # Walk document tree for UserAndSiteMention nodes
    stack = [doc]
    while stack:
        node = stack.pop()
        if isinstance(node, dict):
            if node.get("type") == "UserAndSiteMention":
                attrs = node.get("attrs", {})
                mention_records.append({
                    "comment_id": c["id"],
                    "mention_id": attrs.get("id"),
                    "mention_label": attrs.get("label"),
                    "mention_type": attrs.get("type"),
                })
            for v in node.values():
                stack.append(v)
        elif isinstance(node, list):
            stack.extend(node)

print(f"Total inline mentions extracted: {len(mention_records)}")
print(f"Comments with at least one mention: {len(set(m['comment_id'] for m in mention_records))}/{n}")

# Type distribution
mention_type_counts = Counter(m["mention_type"] for m in mention_records)
print(f"\nMention type distribution:")
for k, v in mention_type_counts.most_common():
    print(f"  {k}: {v}")

In [ ]:
# Mention ID overlap with known entities
user_mention_ids = set(m["mention_id"] for m in mention_records if m["mention_type"] == "user" and m["mention_id"])
site_mention_ids = set(m["mention_id"] for m in mention_records if m["mention_type"] == "site" and m["mention_id"])

print(f"Unique user mention IDs: {len(user_mention_ids)}")
print(f"  Overlap with People: {len(user_mention_ids & people_ids)}/{len(user_mention_ids)}")
missing_user = user_mention_ids - people_ids
if missing_user:
    print(f"  Missing from People: {len(missing_user)} — sample: {list(missing_user)[:5]}")

print(f"\nUnique site mention IDs: {len(site_mention_ids)}")
print(f"  Overlap with Site: {len(site_mention_ids & site_ids)}/{len(site_mention_ids)}")
missing_site = site_mention_ids - site_ids
if missing_site:
    print(f"  Missing from Site: {len(missing_site)} — sample: {list(missing_site)[:5]}")

# Top mentioned users
user_mention_freq = Counter(m["mention_id"] for m in mention_records if m["mention_type"] == "user")
print("\nTop 10 most-mentioned users:")
for mid, cnt in user_mention_freq.most_common(10):
    label = next((m["mention_label"] for m in mention_records if m["mention_id"] == mid), "?")
    in_people = "✓" if mid in people_ids else "✗"
    print(f"  {label:<35} {cnt:>4} mentions  People={in_people}")

## 8 · Cross-Entity Reference Checks (FK Validation)

In [ ]:
# Load normalised entity IDs
def load_norm_ids(filename):
    path = NORMALIZED_DIR / filename
    ids = set()
    if not path.exists():
        print(f"  WARNING: {path} does not exist")
        return ids
    with open(path) as f:
        for line in f:
            row = json.loads(line)
            if row.get("id"):
                ids.add(row["id"])
    return ids

norm_people = load_norm_ids("people.jsonl")
norm_sites = load_norm_ids("sites.jsonl")
norm_contents = load_norm_ids("contents.jsonl")
norm_files = load_norm_ids("files.jsonl")
norm_posts = load_norm_ids("posts.jsonl")
norm_recognitions = load_norm_ids("recognitions.jsonl")
norm_feeds = norm_posts | norm_recognitions

print("Normalised entity counts:")
print(f"  People       : {len(norm_people)}")
print(f"  Site         : {len(norm_sites)}")
print(f"  Content      : {len(norm_contents)}")
print(f"  File         : {len(norm_files)}")
print(f"  Post         : {len(norm_posts)}")
print(f"  Recognition  : {len(norm_recognitions)}")

In [ ]:
# FK field checks
fk_checks = [
    ("feed_comment_author_id", "People", norm_people),
    ("feed_comment_site_id", "Site", norm_sites),
    ("feed_comment_content_id", "Content", norm_contents),
    ("feed_comment_feed_id", "Post|Recognition (normalised)", norm_feeds),
    ("feed_comment_feed_id", "Feed (raw)", raw_feed_ids),
]

print("=== FK Overlap Summary ===\n")
for field, label, ref_ids in fk_checks:
    vals = set(c[field] for c in feed_comments if c.get(field))
    overlap = vals & ref_ids
    missing = vals - ref_ids
    print(f"{field} → {label}")
    print(f"  Unique values : {len(vals)}")
    print(f"  Overlap       : {len(overlap)}")
    print(f"  Missing       : {len(missing)}")
    if missing and len(missing) <= 10:
        print(f"  Missing IDs   : {sorted(missing)[:10]}")
    print()

In [ ]:
# Breakdown: which feed_comment_feed_ids map to Post vs Recognition vs unmatched?
fc_feed_ids = set(c["feed_comment_feed_id"] for c in feed_comments if c.get("feed_comment_feed_id"))
to_post = fc_feed_ids & norm_posts
to_rec = fc_feed_ids & norm_recognitions
to_raw_only = (fc_feed_ids & raw_feed_ids) - norm_feeds
unmatched = fc_feed_ids - raw_feed_ids

print("feed_comment_feed_id breakdown:")
print(f"  Total unique        : {len(fc_feed_ids)}")
print(f"  → normalised Post   : {len(to_post)}")
print(f"  → normalised Recog. : {len(to_rec)}")
print(f"  → raw feed only     : {len(to_raw_only)}  (feed exists but was filtered during normalisation)")
print(f"  → unmatched         : {len(unmatched)}  (no raw feed record found)")

## 9 · Author Analysis

In [ ]:
# Comment count per author
author_counts = Counter(c.get("feed_comment_author_id") for c in feed_comments if c.get("feed_comment_author_id"))
print(f"Unique authors: {len(author_counts)}\n")

print("Top 20 commenters:")
for aid, cnt in author_counts.most_common(20):
    name = next((c.get("feed_comment_author_name") for c in feed_comments if c.get("feed_comment_author_id") == aid), "?")
    in_people = "✓" if aid in norm_people else "✗"
    print(f"  {name:<35} {cnt:>4} comments  People={in_people}")

# Authors with image URL
with_image = sum(1 for c in feed_comments if c.get("feed_comment_author_image_url") not in (None, ""))
print(f"\nAuthors with image URL: {with_image}/{n} records")

# Authors without a name (but have an ID)
no_name = sum(1 for c in feed_comments if c.get("feed_comment_author_id") and not c.get("feed_comment_author_name"))
print(f"Records with author_id but no author_name: {no_name}")

## 10 · Site & Content Context Analysis

In [ ]:
# Comments WITH vs WITHOUT site context
with_site = [c for c in feed_comments if c.get("feed_comment_site_id")]
without_site = [c for c in feed_comments if not c.get("feed_comment_site_id")]
print(f"Comments with site context   : {len(with_site)}/{n} ({len(with_site)/n:.1%})")
print(f"Comments without site context: {len(without_site)}/{n}")

# Site type breakdown
site_type_counts = Counter(c.get("feed_comment_site_type") for c in with_site)
print(f"\nfeed_comment_site_type (among those with site):")
for k, v in site_type_counts.most_common():
    print(f"  {k}: {v}")

# Top sites by comment count
site_comment_counts = Counter(c.get("feed_comment_site_id") for c in with_site)
print(f"\nTop 10 sites by comment count:")
for sid, cnt in site_comment_counts.most_common(10):
    name = next((c.get("feed_comment_site_name") for c in feed_comments if c.get("feed_comment_site_id") == sid), "?")
    print(f"  {name:<40} {cnt:>4} comments")

In [ ]:
# Comments WITH vs WITHOUT content context
with_content = [c for c in feed_comments if c.get("feed_comment_content_id")]
without_content = [c for c in feed_comments if not c.get("feed_comment_content_id")]
print(f"Comments with content context   : {len(with_content)}/{n} ({len(with_content)/n:.1%})")
print(f"Comments without content context: {len(without_content)}/{n}")

# Content type breakdown
content_type_counts = Counter(c.get("feed_comment_content_type") for c in with_content)
print(f"\nfeed_comment_content_type:")
for k, v in content_type_counts.most_common():
    print(f"  {k}: {v}")

# Top content pages by comment count
content_comment_counts = Counter(c.get("feed_comment_content_id") for c in with_content)
print(f"\nTop 10 content pages by comment count:")
for cid, cnt in content_comment_counts.most_common(10):
    title = next((c.get("feed_comment_content_title") for c in feed_comments if c.get("feed_comment_content_id") == cid), "?")
    print(f"  {title[:50]:<50} {cnt:>4} comments")

In [ ]:
# Co-occurrence: site + content present together?
both = sum(1 for c in feed_comments if c.get("feed_comment_site_id") and c.get("feed_comment_content_id"))
site_only = sum(1 for c in feed_comments if c.get("feed_comment_site_id") and not c.get("feed_comment_content_id"))
content_only = sum(1 for c in feed_comments if not c.get("feed_comment_site_id") and c.get("feed_comment_content_id"))
neither = sum(1 for c in feed_comments if not c.get("feed_comment_site_id") and not c.get("feed_comment_content_id"))

print("Site + Content co-occurrence:")
print(f"  Both present    : {both}")
print(f"  Site only       : {site_only}")
print(f"  Content only    : {content_only}")
print(f"  Neither         : {neither}")

## 11 · Comment-per-Feed Distribution

In [ ]:
# How many comments does each feed get?
comments_per_feed = Counter(c.get("feed_comment_feed_id") for c in feed_comments)

print(f"Unique feeds with comments: {len(comments_per_feed)}")
print(f"Total comments: {n}")
print(f"Average comments per feed: {n / len(comments_per_feed):.1f}")

dist = Counter(comments_per_feed.values())
print("\nComments-per-feed distribution:")
for num_comments, num_feeds in sorted(dist.items()):
    print(f"  {num_comments} comment(s) : {num_feeds} feeds")

In [ ]:
# Top feeds by comment count
print("Top 15 feeds by comment count:")
for fid, cnt in comments_per_feed.most_common(15):
    in_norm = "norm" if fid in norm_feeds else ("raw" if fid in raw_feed_ids else "MISS")
    feed_type = "post" if fid in norm_posts else ("recognition" if fid in norm_recognitions else "?")
    print(f"  {fid}  {cnt:>3} comments  [{in_norm}/{feed_type}]")

## 12 · Denormalised Fields — Consistency Check

The raw data carries denormalised copies of site/content metadata on each comment.
Check whether these are consistent (same site_name for same site_id, etc.).

In [ ]:
# Consistency: same site_id → same site_name?
site_id_to_names = defaultdict(set)
site_id_to_types = defaultdict(set)
site_id_to_active = defaultdict(set)

for c in feed_comments:
    sid = c.get("feed_comment_site_id")
    if not sid:
        continue
    site_id_to_names[sid].add(c.get("feed_comment_site_name"))
    site_id_to_types[sid].add(c.get("feed_comment_site_type"))
    site_id_to_active[sid].add(c.get("feed_comment_site_is_active"))

conflicts_name = {sid: names for sid, names in site_id_to_names.items() if len(names) > 1}
conflicts_type = {sid: types for sid, types in site_id_to_types.items() if len(types) > 1}
conflicts_active = {sid: vals for sid, vals in site_id_to_active.items() if len(vals) > 1}

print(f"site_id → site_name conflicts: {len(conflicts_name)}")
print(f"site_id → site_type conflicts: {len(conflicts_type)}")
print(f"site_id → site_is_active conflicts: {len(conflicts_active)}")

if conflicts_name:
    print("\nName conflicts:")
    for sid, names in list(conflicts_name.items())[:5]:
        print(f"  {sid}: {names}")

In [ ]:
# Consistency: same content_id → same content_title / content_type?
content_id_to_titles = defaultdict(set)
content_id_to_types = defaultdict(set)

for c in feed_comments:
    cid = c.get("feed_comment_content_id")
    if not cid:
        continue
    content_id_to_titles[cid].add(c.get("feed_comment_content_title"))
    content_id_to_types[cid].add(c.get("feed_comment_content_type"))

conflicts_title = {cid: titles for cid, titles in content_id_to_titles.items() if len(titles) > 1}
conflicts_ctype = {cid: types for cid, types in content_id_to_types.items() if len(types) > 1}

print(f"content_id → content_title conflicts: {len(conflicts_title)}")
print(f"content_id → content_type conflicts: {len(conflicts_ctype)}")

if conflicts_title:
    print("\nTitle conflicts:")
    for cid, titles in list(conflicts_title.items())[:5]:
        print(f"  {cid}: {titles}")

## 13 · Summary & Observations

Run all cells above, then fill in your observations here.

In [ ]:
# Final summary table
summary = {
    "Total records": n,
    "Unique comment IDs": len(set(c["id"] for c in feed_comments)),
    "Unique authors": len(set(c.get("feed_comment_author_id") for c in feed_comments if c.get("feed_comment_author_id"))),
    "Unique feeds referenced": len(set(c.get("feed_comment_feed_id") for c in feed_comments if c.get("feed_comment_feed_id"))),
    "  → matched normalised Post|Recognition": len(fc_feed_ids & norm_feeds),
    "  → matched raw feed only": len(to_raw_only),
    "  → unmatched": len(unmatched),
    "Unique sites referenced": len(set(c.get("feed_comment_site_id") for c in feed_comments if c.get("feed_comment_site_id"))),
    "Unique contents referenced": len(set(c.get("feed_comment_content_id") for c in feed_comments if c.get("feed_comment_content_id"))),
    "Records with file attachments": len(files_nonempty),
    "Records with audiences": len(aud_nonempty),
    "Inline mentions (total)": len(mention_records),
    "  → user mentions": sum(1 for m in mention_records if m["mention_type"] == "user"),
    "  → site mentions": sum(1 for m in mention_records if m["mention_type"] == "site"),
    "is_deleted == True": sum(1 for c in feed_comments if c.get("is_deleted") is True),
}

print("=" * 60)
print("FEED COMMENT EXPLORATION SUMMARY")
print("=" * 60)
for k, v in summary.items():
    print(f"  {k:<45} {v}")

## 14 · Candidate Ontology (to be finalised)

Based on the exploration above, here is the proposed graph model:

### Node: `FeedComment`
**Properties:**
- `id` (PK)
- `createddate`
- `lastmodifieddate`
- `is_deleted`
- `feed_comment_ref_data` (plain text content)
- `feed_comment_html_data` (HTML content)
- `feed_comment_content_type` (nullable — page, album, event)
- `feed_comment_site_type` (nullable — public, private)

### Relationships
| Relationship | From | To | Key field | Coverage |
|---|---|---|---|---|
| `COMMENTED_ON` | `FeedComment` | `Post` or `Recognition` | `feed_comment_feed_id` | 100 % present, ~78 % normalised |
| `AUTHORED` | `People` | `FeedComment` | `feed_comment_author_id` | 100 % present, 100 % People match |
| `BELONGS_TO_SITE` | `FeedComment` | `Site` | `feed_comment_site_id` | 69 % present, 100 % Site match |
| `BELONGS_TO_CONTENT` | `FeedComment` | `Content` | `feed_comment_content_id` | 51 % present, 100 % Content match |
| `MENTIONS` | `FeedComment` | `People` | inline user mentions | ~846 mentions |
| `MENTIONS` | `FeedComment` | `Site` | inline site mentions | ~12 mentions |
| `HAS_FILE` | `FeedComment` | `File` | `feed_comment_file[].file_id` | 65 records |

### Deferred / Low-priority
- `feed_audiences` — audience IDs (233 records); entity type TBD
- `feed_comment_deleted_files` — only 1 record has data
- Denormalised metadata (author_name, site_name, content_title, thumbnails) — use as fallback properties, not FKs